# Pretraining notebook

## Section 1: Generate text before training

Load your current model

Enter a few prompts and note the output.

Try these:
 - love is
 - baby
 - tonight

Generate 20 tokens and note the output.

Reflection: Is there any quality or meaning to the output or is it largely random text?  Why?

In [1]:
import sys
import torch

print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: d:\Fall 2026\CS 397 LLM_REASONING\WeirdAIStarter\.venv\Scripts\python.exe
Torch: 2.14.0+cu126
CUDA version: 12.6
CUDA available: True
GPU: NVIDIA T1000 8GB


In [2]:
import torch

# This makes results more reproducible and reduces confusion when comparing outputs.
torch.manual_seed(123)

from weird_ai.model import WeirdAIModel

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

# TODO:
# Load your model

from weird_ai.tokenizer import SimpleCharacterTokenizer
from weird_ai.config import SAMPLE_LYRICS_FILE

text = SAMPLE_LYRICS_FILE.read_text(encoding="utf-8")

tokenizer = SimpleCharacterTokenizer(text)

vocab_size = len(tokenizer.stoi)

model = WeirdAIModel(
    vocab_size=vocab_size
)

model = model.to(device)

context_size = model.context_length


# TODO:
# Generate text from:
# "love is"
# "baby"
# "tonight"

from weird_ai.generation import (
    text_to_token_ids,
    token_ids_to_text,
    generate_text_simple
)

prompts = [
    "love is",
    "baby",
    "tonight"
]

model.eval()

for prompt in prompts:
    input_ids = text_to_token_ids(
        prompt,
        tokenizer
    ).to(device)

    output_ids = generate_text_simple(
        model=model,
        input_ids=input_ids,
        max_new_tokens=20,
        context_size=context_size
    )

    output_text = token_ids_to_text(
        output_ids,
        tokenizer
    )

    print(f"Prompt: {prompt}")
    print(f"Output: {output_text}")
    print()

cuda
Prompt: love is
Output: love is빨세对부bt天V思모د패
а虹러타추핑藍

Prompt: baby
Output: baby린h었필행对부bt天V思모د패
а虹뻐Ü

Prompt: tonight
Output: tonight«很а부bt天억で모د패
а虹뻐Ü널倒í



## Section 2: Understanding Logits

Using the logits starter code, do the following:
 - Apply softmax
 - Identify the highest probability token

Reflection: Why is softmax necessary before interpreting logits as probabilities?

In [3]:
logits = torch.tensor([
    [1.5, 2.0, 0.5]
])

probabilities = torch.softmax(logits, dim=-1)

print(probabilities)
print(probabilities.sum())

predicted_token = torch.argmax(probabilities)

print(predicted_token)

tensor([[0.3315, 0.5465, 0.1220]])
tensor(1.0000)
tensor(1)


## Section 3: Cross Entropy Loss

### Textbook 
The textbook discussion begins on page 136

### Manually calculate
 - Probabilities
 - Log probabilities
 - Average negative log probabilities

### Compare with torch
Compare your manual results to the torch cross entropy results

```python
     torch.nn.functional.cross_entropy(...)
'''


In [4]:
import torch

# Manually calculate values

probs = torch.tensor([
    0.7,
    0.2,
    0.1
])

target_index = 0

In [5]:
target_probability = probs[target_index]

print(target_probability)

tensor(0.7000)


In [6]:
log_probability = torch.log(target_probability)

print(log_probability)

tensor(-0.3567)


In [7]:
loss = -log_probability

print(loss)

tensor(0.3567)


In [8]:
# Compare with torch calculations
logits = torch.log(probs).unsqueeze(0)
target = torch.tensor([target_index])

torch_loss = torch.nn.functional.cross_entropy(
    logits,
    target
)

print(torch_loss)

tensor(0.3567)


## Section 4: Perplexity

Compute the perplexity

```python
perplexity = torch.exp(loss)
```

**Note:** A perplexity of 10 means the model is roughly as uncertain as choosing among 10 equally likely next tokens.

In [9]:
loss = torch.tensor(2.5)

perplexity = torch.exp(loss)

print(perplexity)

tensor(12.1825)


## Section 5: Understanding the Training Loop

The training loop performs:

1. Iterate through epochs
2. Iterate through batches
3. Zero gradients
4. Calculate loss
5. Backpropagation
6. Optimizer step
7. Evaluate model
8. Generate sample text

Draw or explain this process in your own words.

# Training vs. Validation

You will:
 - Split lyrics into train/validation
 - Create loaders
 - Compute initial loss

Keep note of your: 
 - Training Loss: 
 - Validation Loss: 

In [10]:
from weird_ai.dataset import LyricsDataset

train_ratio = 0.9

# Split the corpus into training and validation text
split_index = int(len(text) * train_ratio)

train_text = text[:split_index]
val_text = text[split_index:]

print("Training characters:", len(train_text))
print("Validation characters:", len(val_text))

Training characters: 12156964
Validation characters: 1350774


In [11]:
from torch.utils.data import DataLoader, Subset

# Convert text into tokens
train_tokens = tokenizer.encode(train_text)
val_tokens = tokenizer.encode(val_text)

# Create datasets
train_dataset = LyricsDataset(
    train_tokens,
    block_size=context_size
)

val_dataset = LyricsDataset(
    val_tokens,
    block_size=context_size
)

# Use non-overlapping starting positions
train_subset = Subset(
    train_dataset,
    range(0, len(train_dataset), context_size)
)

val_subset = Subset(
    val_dataset,
    range(0, len(val_dataset), context_size)
)

# Create dataloaders
train_loader = DataLoader(
    train_subset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=32,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training batches: 2968
Validation batches: 330


In [12]:
from weird_ai.losses import calc_loss_loader

train_loss = calc_loss_loader(
    train_loader,
    model,
    device,
    num_batches=5
)

val_loss = calc_loss_loader(
    val_loader,
    model,
    device,
    num_batches=5
)

print("Training Loss:", train_loss)
print("Validation Loss:", val_loss)

Training Loss: 6.69064302444458
Validation Loss: 6.686367034912109


## Section 6: Training

Train for one Epoch
```python
train_model(...)
```

Then record the loss before and after training.  

In [13]:
num_epochs = 1

# TODO:
# Train the model

from weird_ai.trainer import train_model_simple

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0004
)

initial_loss = train_loss

train_losses, val_losses, tokens_seen = train_model_simple(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    num_epochs=num_epochs,
    eval_freq=100,
    eval_iter=5,
    start_context="love is",
    tokenizer=tokenizer,
    context_size=context_size
)

# TODO:
# Record:
# Training loss before
# Training loss after

final_loss = calc_loss_loader(
    train_loader,
    model,
    device,
    num_batches=5
)

Epoch 1, Step 0: Train loss 6.603, Validation loss 6.599
Epoch 1, Step 100: Train loss 2.863, Validation loss 2.849
Epoch 1, Step 200: Train loss 2.633, Validation loss 2.650
Epoch 1, Step 300: Train loss 2.578, Validation loss 2.584
Epoch 1, Step 400: Train loss 2.490, Validation loss 2.520
Epoch 1, Step 500: Train loss 2.420, Validation loss 2.440
Epoch 1, Step 600: Train loss 2.324, Validation loss 2.353
Epoch 1, Step 700: Train loss 2.247, Validation loss 2.274
Epoch 1, Step 800: Train loss 2.165, Validation loss 2.208
Epoch 1, Step 900: Train loss 2.139, Validation loss 2.159
Epoch 1, Step 1000: Train loss 2.082, Validation loss 2.124
Epoch 1, Step 1100: Train loss 2.042, Validation loss 2.095
Epoch 1, Step 1200: Train loss 2.045, Validation loss 2.073
Epoch 1, Step 1300: Train loss 1.971, Validation loss 2.048
Epoch 1, Step 1400: Train loss 1.985, Validation loss 2.030
Epoch 1, Step 1500: Train loss 1.967, Validation loss 2.013
Epoch 1, Step 1600: Train loss 1.930, Validation los

In [14]:
print(f"Initial loss: {initial_loss}")
print(f"Final loss: {final_loss}")

Initial loss: 6.69064302444458
Final loss: 1.766219162940979


## Evaluation

Did the loss decrease?

Why is decreasing loss important?

Yes, the loss decreased from about 6.69 to 1.77 after training. Decreasing loss is important because it shows that the model is learning from the training data and becoming better at predicting the next token.

## Before and After Comparison

Prompt:

love is

Before Training:
__________________

After Training:
__________________

Reflection:

How did the output change?

What evidence do you see that the model learned something from the training data?

The output stopped using unrelated characters that were in a different language. Also while the lyric is not gramattically correct,
it is getting closer. The evidence that I see is that the output is using only english characters and the loss decreasing from almost 7 to about 2. 


In [15]:
prompt = "love is"

input_ids = text_to_token_ids(
    prompt,
    tokenizer
).to(device)

output_ids = generate_text_simple(
    model=model,
    input_ids=input_ids,
    max_new_tokens=20,
    context_size=context_size
)

output_text = token_ids_to_text(
    output_ids,
    tokenizer
)

print(output_text)

love is a be the been the s


Pre-trained Weird AI Checkpoint

In [16]:
model.eval()

prompt = "love is"

input_ids = text_to_token_ids(
    prompt,
    tokenizer
).to(device)

trained_output_ids = generate_text_simple(
    model=model,
    input_ids=input_ids,
    max_new_tokens=20,
    context_size=context_size
)

trained_output_text = token_ids_to_text(
    trained_output_ids,
    tokenizer
)

print("Output immediately after training:")
print(trained_output_text)

Output immediately after training:
love is a be the been the s


In [17]:
from pathlib import Path
from weird_ai.trainer import save_checkpoint, load_checkpoint

# Find the WeirdAIStarter project root
cwd = Path.cwd()

if (cwd / "pyproject.toml").exists():
    project_root = cwd
elif (cwd.parent / "pyproject.toml").exists():
    project_root = cwd.parent
else:
    raise RuntimeError("Could not find the WeirdAIStarter project root.")

checkpoint_path = (
    project_root
    / "models"
    / "lesson-05-pretrained"
    / "checkpoint.pt"
)

save_checkpoint(
    model=model,
    optimizer=optimizer,
    epoch=num_epochs,
    train_losses=train_losses,
    val_losses=val_losses,
    track_tokens_seen=tokens_seen,
    checkpoint_path=checkpoint_path
)

print("Checkpoint saved to:")
print(checkpoint_path)

print("Checkpoint exists:", checkpoint_path.exists())

Checkpoint saved to:
d:\Fall 2026\CS 397 LLM_REASONING\WeirdAIStarter\models\lesson-05-pretrained\checkpoint.pt
Checkpoint exists: True
